# 🍃 PRANA ETL Pipeline — Tab 1: Ikhtisar Nasional (Executive Cockpit)

Notebook ini bertugas mengekstrak, membersihkan, menyelaraskan unit pengukuran, mengagregasi, serta mengekspor data dari **27 berkas AQMS telemetri**, **data surveilans rawat inap rumah sakit**, dan **tabel master geospasial** ke dalam satu berkas terstruktur:  
📁 **`dataset/dashboard/tab1_ikhtisar.json`**

---

### 🎯 Tujuan Analisis & Visualisasi Tab 1:
1. **Executive KPI Cockpit (4 Metrik Utama):**
   - **Rata-rata PM2.5 Nasional (µg/m³)**: Mengukur konsentrasi rata-rata partikulat halus di seluruh wilayah pemantauan.
   - **Persentase Hari Melebihi Batas 24 Jam (% Hari > 55 µg/m³)**: Tingkat pelanggaran baku mutu PP 22/2021.
   - **Stasiun Paling Kritis**: Menemukan lokasi dengan paparan polutan tertinggi secara berkelanjutan.
   - **Hospital Burden Index (%)**: Rasio estimasi beban admisi rawat inap penyakit pernapasan harian terhadap daya tampung tempat tidur rumah sakit rujukan.
2. **Distribusi ISPU Dominan (Donut Chart)**: Proporsi stasiun berdasarkan kategori dominan tahunan (Baik, Sedang, Tidak Sehat, Sangat Tidak Sehat, Berbahaya).
3. **Peringkat Top 10 & Bottom 10 Stasiun (Horizontal Bar Chart)**: Peringkat kualitas udara terburuk dan terbaik.
4. **Peta Sebaran 27 Stasiun AQMS (Interactive Leaflet Map)**: Titik koordinat stasiun dengan status warna ISPU interaktif.

---

### 📐 Standar Regulasi & Konversi yang Diterapkan:
- **Permen LHK No. 14 Tahun 2020** (ISPU PM2.5 Rata-rata 24 Jam):
  - 🟢 **Baik**: $0 - 15.0\ \mu\text{g/m}^3$
  - 🔵 **Sedang**: $15.1 - 55.0\ \mu\text{g/m}^3$
  - 🟠 **Tidak Sehat**: $55.1 - 150.0\ \mu\text{g/m}^3$
  - 🔴 **Sangat Tidak Sehat**: $150.1 - 250.0\ \mu\text{g/m}^3$
  - ⚫ **Berbahaya**: $> 250.0\ \mu\text{g/m}^3$
- **PP No. 22 Tahun 2021**:
  - Baku mutu 24 jam: $55\ \mu\text{g/m}^3$
  - Baku mutu tahunan: $15\ \mu\text{g/m}^3$
- **WHO 2021 Guideline**:
  - Ambang batas tahunan: $5\ \mu\text{g/m}^3$

In [1]:
# =============================================================================
# LANGKAH 0: Inisialisasi Environment, Library, dan Path Direktori
# =============================================================================
# Memuat pustaka inti data science (pandas untuk tabulasi, numpy untuk kalkulasi numerik)
import os, glob, json, time
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

# Definisi path proyek berbasis Pathlib agar kompatibel di Windows maupun Linux/macOS
PROJECT_ROOT   = Path(r'd:/GLRS')
AQMS_DIR       = PROJECT_ROOT / 'dataset/raw/aqms'
HEALTH_FILE    = PROJECT_ROOT / 'dataset/raw/health/HLT-01_health_outcomes_daily.csv'
SITES_FILE     = PROJECT_ROOT / 'dataset/raw/master/sites.csv'
HOSPITALS_FILE = PROJECT_ROOT / 'dataset/raw/master/hospitals.csv'
SENSORS_FILE   = PROJECT_ROOT / 'dataset/raw/master/sensors.csv'

# Direktori target JSON untuk frontend dashboard
OUT_DIR        = PROJECT_ROOT / 'dataset/dashboard'
OUT_FILE       = OUT_DIR / 'tab1_ikhtisar.json'

# Buat folder output bila belum tersedia
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Verifikasi keberadaan file AQMS
aqms_files = sorted(list(AQMS_DIR.glob('*.csv')))
print('✅ Environment & Path siap!')
print(f'   Ditemukan {len(aqms_files)} berkas AQMS CSV di direktori: {AQMS_DIR}')

✅ Environment & Path siap!
   Ditemukan 27 berkas AQMS CSV di direktori: d:\GLRS\dataset\raw\aqms


In [2]:
# =============================================================================
# LANGKAH 1: Memuat Data Master (Sites, Sensors, & Hospitals)
# =============================================================================
# Tabel master memuat metadata stasiun (koordinat, nama sensor, kelas rumah sakit)
sites     = pd.read_csv(SITES_FILE)
hospitals = pd.read_csv(HOSPITALS_FILE)
sensors   = pd.read_csv(SENSORS_FILE)

# Catatan Penting Data Engineering:
# 'sensors.csv' dan 'sites.csv' sama-sama memiliki kolom 'location_id' dan 'region_group'.
# Agar tidak menghasilkan kolom duplikat '_x' dan '_y', kita pilih kolom secara eksplisit
# lalu menggabungkannya berdasarkan kunci 'location_id'.
sensor_site = sensors[
    ['sensor_id', 'location_id', 'nama_sensor', 'model_sensor', 'status_operasional']
].merge(
    sites[[
        'location_id', 'station_code', 'kota_kabupaten', 'provinsi', 'kecamatan',
        'karakteristik_wilayah', 'latitude_dd', 'longitude_dd', 'region_group',
        'baseline_pm25', 'site_type'
    ]],
    on='location_id',
    how='left'
)

print('✅ Data Master Berhasil Digabungkan:')
print(f'   Total Lokasi Stasiun : {len(sites)} stasiun')
print(f'   Total Sensor Terdaftar: {len(sensors)} sensor')
print(f'   Total Rumah Sakit     : {len(hospitals)} faskes rujukan')

# Pratinjau 3 baris pertama data relasi sensor & lokasi
sensor_site[['sensor_id', 'station_code', 'nama_sensor', 'kota_kabupaten', 'region_group']].head(3)

✅ Data Master Berhasil Digabungkan:
   Total Lokasi Stasiun : 27 stasiun
   Total Sensor Terdaftar: 27 sensor
   Total Rumah Sakit     : 27 faskes rujukan


,sensor_id,station_code,nama_sensor,kota_kabupaten,region_group
0,SENS-001,AQMS-JKT-01,AQMS Gambir Medan Merdeka,Kota Administrasi Jakarta Pusat,Jakarta
1,SENS-002,AQMS-JKT-02,AQMS Marunda Industrial Port,Kota Administrasi Jakarta Utara,Jakarta
2,SENS-003,AQMS-JKT-03,AQMS Cengkareng Daan Mogot Artery,Kota Administrasi Jakarta Barat,Jakarta


In [3]:
# =============================================================================
# LANGKAH 2: Membaca & Menyelaraskan (Normalize) 27 Berkas Telemetri AQMS
# =============================================================================
# Anomali Dataset:
# - 24 berkas menggunakan header standar: 'pm25_ug_m3', 'pm10_ug_m3'
# - 3 berkas (BJB-01, MDN-01, PSW-01) mengekspor data dalam logger 'ppm' (pm25_ppm, pm10_ppm).
#   Sesuai metadata sensor, nilai 'ppm' ini adalah kelipatan 1/1000 dari µg/m³,
#   sehingga harus dikalikan 1.000 agar setara dalam satuan standar µg/m³.

t_start = time.time()
dfs = []

for f in aqms_files:
    df = pd.read_csv(f, low_memory=False)
    
    # Penyelarasan PM2.5: ubah teks ke numerik; jika berformat ppm, konversi ke µg/m³
    if 'pm25_ppm' in df.columns:
        df['pm25_ug_m3'] = pd.to_numeric(df['pm25_ppm'], errors='coerce') * 1000.0
    elif 'pm25_ug_m3' in df.columns:
        df['pm25_ug_m3'] = pd.to_numeric(df['pm25_ug_m3'], errors='coerce')
    
    # Penyelarasan PM10 (untuk kebutuhan audit komparatif)
    if 'pm10_ppm' in df.columns:
        df['pm10_ug_m3'] = pd.to_numeric(df['pm10_ppm'], errors='coerce') * 1000.0
    elif 'pm10_ug_m3' in df.columns:
        df['pm10_ug_m3'] = pd.to_numeric(df['pm10_ug_m3'], errors='coerce')
    
    # Pilih hanya kolom yang dibutuhkan untuk efisiensi memori
    cols_needed = ['measurement_id', 'sensor_id', 'timestamp_start', 'pm25_ug_m3', 'pm10_ug_m3', 'qc_flag']
    available_cols = [c for c in cols_needed if c in df.columns]
    dfs.append(df[available_cols])

# Satukan seluruh baris dari 27 berkas ke dalam satu DataFrame raksasa
raw_aqms = pd.concat(dfs, ignore_index=True)

# Bersihkan anomali nilai 'inf' atau '-inf' yang muncul akibat kesalahan sensor (misal di JMB-02)
raw_aqms['pm25_ug_m3'] = raw_aqms['pm25_ug_m3'].replace([np.inf, -np.inf], np.nan)

elapsed = time.time() - t_start
print(f'✅ Berhasil membaca & menstandardisasi 27 berkas AQMS dalam {elapsed:.2f} detik!')
print(f'   Total Baris Data Mentah : {len(raw_aqms):,} observasi per jam')
print(f'   Sensor Terwakili        : {raw_aqms["sensor_id"].nunique()} dari 27 sensor')

✅ Berhasil membaca & menstandardisasi 27 berkas AQMS dalam 0.92 detik!
   Total Baris Data Mentah : 667,981 observasi per jam
   Sensor Terwakili        : 27 dari 27 sensor


In [4]:
# =============================================================================
# LANGKAH 3: Parsing Waktu & Filter Data Berkualitas Tinggi (QC_FLAG = VALID)
# =============================================================================
# Catatan Teknis Waktu:
# Format timestamp di berkas ada yang memiliki mikrodetik (misal .420) dan ada yang bulat (:00).
# Menggunakan format='mixed' memastikan pandas mengurai semua variasi string tanpa menghasilkan NaT.
raw_aqms['ts'] = pd.to_datetime(raw_aqms['timestamp_start'], format='mixed', errors='coerce')

# Jendela Observasi Kasus Studi Kompetisi:
# 1 Januari 2024 s.d. 14 Oktober 2026
DATE_START = pd.Timestamp('2024-01-01')
DATE_END   = pd.Timestamp('2026-10-14 23:59:59')

in_window = raw_aqms[(raw_aqms['ts'] >= DATE_START) & (raw_aqms['ts'] <= DATE_END)].copy()

# Filter Kontrol Kualitas (QC Flag):
# Sesuai dokumen kasus studi Bab 4, data utama hanya menggunakan status 'VALID'.
# Status 'SUSPECT', 'COMM_LOST', 'GARBAGE', dll dikeluarkan dari perhitungan utama.
valid_aqms = in_window[in_window['qc_flag'] == 'VALID'].copy()

# Tambahkan kolom tanggal (date) untuk agregasi harian
valid_aqms['date'] = valid_aqms['ts'].dt.date

pct_valid_all = (len(valid_aqms) / len(in_window)) * 100
print(f'✅ Filter Kualitas Selesai:')
print(f'   Data dalam rentang waktu : {len(in_window):,} baris')
print(f'   Data berstatus VALID     : {len(valid_aqms):,} baris ({pct_valid_all:.1f}%)')
print(f'   Sensor Aktif             : {valid_aqms["sensor_id"].nunique()} sensor')
print(f'   Rentang Tanggal Efektif  : {valid_aqms["ts"].min()} s.d. {valid_aqms["ts"].max()}')

✅ Filter Kualitas Selesai:
   Data dalam rentang waktu : 667,844 baris
   Data berstatus VALID     : 599,179 baris (89.7%)
   Sensor Aktif             : 27 sensor
   Rentang Tanggal Efektif  : 2024-01-01 00:00:00 s.d. 2026-10-14 23:11:22.750000


In [5]:
# =============================================================================
# LANGKAH 4: Agregasi Harian & Klasifikasi Indeks Kualitas Udara (ISPU PM2.5)
# =============================================================================
# Sesuai Permen LHK 14/2020, penetapan ISPU dihitung berdasarkan rata-rata 24 jam.
# Maka kita kelompokkan data per sensor dan per tanggal.
daily_pm25 = (
    valid_aqms.groupby(['sensor_id', 'date'])['pm25_ug_m3']
    .mean()
    .reset_index()
    .rename(columns={'pm25_ug_m3': 'daily_avg_pm25'})
)

# Definisi Ambang Batas ISPU PM2.5 24-jam (Permen LHK 14/2020)
ISPU_BINS   = [0.0, 15.0, 55.0, 150.0, 250.0, np.inf]
ISPU_LABELS = ['Baik', 'Sedang', 'Tidak Sehat', 'Sangat Tidak Sehat', 'Berbahaya']
ISPU_COLORS = {
    'Baik':                '#10b981',  # Hijau Zamrud
    'Sedang':              '#0284c7',  # Biru Langit
    'Tidak Sehat':         '#f59e0b',  # Kuning Amber
    'Sangat Tidak Sehat':  '#ef4444',  # Merah Tua
    'Berbahaya':           '#1e1b4b',  # Hitam Ungu Pekat
}

# Kategorisasi setiap hari pemantauan ke dalam kelas ISPU
daily_pm25['ispu_cat'] = pd.cut(
    daily_pm25['daily_avg_pm25'],
    bins=ISPU_BINS,
    labels=ISPU_LABELS,
    right=True
).astype(str)

# Cek apakah melebihi batas 24-jam baku mutu nasional PP 22/2021 (55 µg/m³)
PP22_LIMIT_24H = 55.0
daily_pm25['exceed_24h'] = daily_pm25['daily_avg_pm25'] > PP22_LIMIT_24H

print('✅ Agregasi Harian ISPU Selesai:')
print(f'   Total Observasi Harian Stasiun: {len(daily_pm25):,} station-days')
print('\nDistribusi Kategori ISPU Harian Nasional:')
print(daily_pm25['ispu_cat'].value_counts())

✅ Agregasi Harian ISPU Selesai:
   Total Observasi Harian Stasiun: 26,330 station-days

Distribusi Kategori ISPU Harian Nasional:
ispu_cat
Sedang                17205
Baik                   4455
Tidak Sehat            3986
Sangat Tidak Sehat      370
Berbahaya               314
Name: count, dtype: int64


In [6]:
# =============================================================================
# LANGKAH 5: Agregasi Tingkat Stasiun (27 Stasiun)
# =============================================================================
# Menghitung metrik performa masing-masing stasiun:
# 1. Rata-rata PM2.5 sepanjang periode (µg/m³)
# 2. Persentase hari yang melanggar batas PP 22/2021 (% hari > 55 µg/m³)
# 3. Kategori ISPU paling sering muncul (dominant category)
# 4. Tingkat kelengkapan data (data completeness / valid %)

# 5a. Rata-rata konsentrasi dan pelanggaran harian
station_stats = (
    daily_pm25.groupby('sensor_id')
    .agg(
        n_days=('date', 'count'),
        n_exceed=('exceed_24h', 'sum'),
        pct_exceed_24h=('exceed_24h', lambda x: round(x.mean() * 100, 1)),
        avg_pm25=('daily_avg_pm25', lambda x: round(x.mean(), 2))
    )
    .reset_index()
)

# 5b. Kategori ISPU dominan (modus)
dominant_ispu = (
    daily_pm25.groupby('sensor_id')['ispu_cat']
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
    .rename(columns={'ispu_cat': 'dominant_ispu_cat'})
)

# 5c. Persentase data valid terhadap total data yang terekam sensor
total_rows_per_sensor = in_window.groupby('sensor_id').size().reset_index(name='n_total_rows')
valid_rows_per_sensor = valid_aqms.groupby('sensor_id').size().reset_index(name='n_valid_rows')
completeness_df = total_rows_per_sensor.merge(valid_rows_per_sensor, on='sensor_id', how='left')
completeness_df['pct_valid'] = (completeness_df['n_valid_rows'] / completeness_df['n_total_rows'] * 100).round(1)

# 5d. Gabungkan seluruh ringkasan stasiun dengan master geospasial
station_df = (
    station_stats
    .merge(dominant_ispu, on='sensor_id', how='left')
    .merge(completeness_df[['sensor_id', 'pct_valid']], on='sensor_id', how='left')
    .merge(sensor_site, on='sensor_id', how='left')
)

# Pasangkan kode warna resmi sesuai kategori dominan
station_df['ispu_color'] = station_df['dominant_ispu_cat'].map(ISPU_COLORS)

# Urutkan peringkat stasiun dari konsentrasi PM2.5 terburuk (tertinggi) ke terbersih
station_df = station_df.sort_values('avg_pm25', ascending=False).reset_index(drop=True)
station_df['rank'] = range(1, len(station_df) + 1)

print(f'✅ Agregasi 27 Stasiun Lengkap!')
print(f'   Stasiun Paling Berpolusi : {station_df.iloc[0]["station_code"]} ({station_df.iloc[0]["kota_kabupaten"]}) - {station_df.iloc[0]["avg_pm25"]} µg/m³')
print(f'   Stasiun Paling Bersih    : {station_df.iloc[-1]["station_code"]} ({station_df.iloc[-1]["kota_kabupaten"]}) - {station_df.iloc[-1]["avg_pm25"]} µg/m³')

✅ Agregasi 27 Stasiun Lengkap!
   Stasiun Paling Berpolusi : AQMS-BJB-01 (Kota Banjarbaru) - 102.14 µg/m³
   Stasiun Paling Bersih    : AQMS-IDP-01 (Kabupaten Aceh Besar) - 9.34 µg/m³


In [7]:
# =============================================================================
# LANGKAH 6: Perhitungan 4 Kartu KPI Utama (Executive Cockpit)
# =============================================================================

# KPI 1: Rata-Rata PM2.5 Nasional (µg/m³)
# Menggambarkan paparan rata-rata tahunan across seluruh 27 stasiun pemantau
avg_pm25_national = round(float(station_df['avg_pm25'].mean()), 2)

# KPI 2: Persentase Hari Melebihi Batas 24 Jam PP 22/2021 (55 µg/m³)
# Proporsi total hari di seluruh stasiun yang melanggar baku mutu
pct_days_exceed_national = round(float(daily_pm25['exceed_24h'].mean() * 100), 1)

# KPI 3: Profil Stasiun Paling Kritis
top_critical = station_df.iloc[0]
station_most_critical = {
    'station_code': str(top_critical['station_code']),
    'station_name': str(top_critical['nama_sensor']),
    'kota':         str(top_critical['kota_kabupaten']),
    'provinsi':     str(top_critical['provinsi']),
    'avg_pm25':     float(top_critical['avg_pm25']),
    'pct_exceed':   float(top_critical['pct_exceed_24h']),
}

# KPI 4: Indeks Beban Rumah Sakit (Hospital Burden Index)
# Dihitung dari rasio rata-rata kunjungan harian penyakit respirasi (balita + lansia)
# terhadap total kapasitas tempat tidur seluruh rumah sakit rujukan.
df_health = pd.read_csv(HEALTH_FILE)
df_health['date_dt'] = pd.to_datetime(df_health['date'], errors='coerce')
df_health_filtered = df_health[
    (df_health['date_dt'] >= DATE_START) & (df_health['date_dt'] <= DATE_END)
].copy()

# Ambil kasus respirasi balita (under5) dan lansia (over60)
under5_cases = pd.to_numeric(df_health_filtered['resp_admissions_under5'], errors='coerce').fillna(0)
over60_cases = pd.to_numeric(df_health_filtered['resp_admissions_over60'], errors='coerce').fillna(0)
total_resp_admissions = under5_cases.sum() + over60_cases.sum()

# Total daya tampung tempat tidur rumah sakit rujukan
bed_capacity = pd.to_numeric(hospitals['kapasitas_tempat_tidur'], errors='coerce').fillna(0).sum()
n_unique_health_days = df_health_filtered['date_dt'].nunique()

# Beban rawat inap harian per total kasur
avg_daily_resp_admissions = total_resp_admissions / n_unique_health_days if n_unique_health_days > 0 else 0
hospital_burden_index = round(float((avg_daily_resp_admissions / bed_capacity) * 100), 2) if bed_capacity > 0 else 0

print('===========================================================')
print('          🏆 RINGKASAN 4 KPI UTAMA (EXECUTIVE COCKPIT)    ')
print('===========================================================')
print(f'1. Rata-Rata PM2.5 Nasional        : {avg_pm25_national} µg/m³')
print(f'2. % Hari Melebihi Batas PP22/2021 : {pct_days_exceed_national}% hari')
print(f'3. Stasiun Paling Kritis           : {station_most_critical["station_code"]} ({station_most_critical["kota"]}) [{station_most_critical["avg_pm25"]} µg/m³]')
print(f'4. Hospital Burden Index           : {hospital_burden_index}%')
print(f'   (Rata-rata {avg_daily_resp_admissions:.1f} pasien respirasi/hari pada {int(bed_capacity):,} kasur faskes)')
print('===========================================================')

          🏆 RINGKASAN 4 KPI UTAMA (EXECUTIVE COCKPIT)    
1. Rata-Rata PM2.5 Nasional        : 43.25 µg/m³
2. % Hari Melebihi Batas PP22/2021 : 17.7% hari
3. Stasiun Paling Kritis           : AQMS-BJB-01 (Kota Banjarbaru) [102.14 µg/m³]
4. Hospital Burden Index           : 6.43%
   (Rata-rata 404.0 pasien respirasi/hari pada 6,280 kasur faskes)


In [8]:
# =============================================================================
# LANGKAH 7: Struktur Data Donut Chart (Distribusi ISPU Dominan Stasiun)
# =============================================================================
# Menghitung berapa banyak stasiun yang dominan berada di kategori Baik, Sedang, dst.
donut_counts = station_df['dominant_ispu_cat'].value_counts()

# Susun ke dalam format list of dictionary yang siap dipakai komponen Chart.js / ECharts
ispu_donut_data = []
for cat in ISPU_LABELS:
    count = int(donut_counts.get(cat, 0))
    if count > 0:  # Hanya sertakan kategori yang memiliki stasiun
        ispu_donut_data.append({
            'category':   cat,
            'count':      count,
            'percentage': round(count / len(station_df) * 100, 1),
            'color':      ISPU_COLORS.get(cat, '#94a3b8')
        })

print('✅ Data Donut Chart:')
for item in ispu_donut_data:
    print(f"   • {item['category']:<18}: {item['count']} stasiun ({item['percentage']}%) [{item['color']}]")

✅ Data Donut Chart:
   • Baik              : 4 stasiun (14.8%) [#10b981]
   • Sedang            : 21 stasiun (77.8%) [#0284c7]
   • Tidak Sehat       : 2 stasiun (7.4%) [#f59e0b]


In [9]:
# =============================================================================
# LANGKAH 8: Data Peringkat Stasiun (Top 10 & Bottom 10 Horizontal Bar Chart)
# =============================================================================
# Menghasilkan list 27 stasiun lengkap yang telah terurut dari PM2.5 tertinggi ke terendah.
# Frontend dapat dengan mudah memotong [:10] untuk Top 10 dan [-10:] untuk Bottom 10.
pm25_ranking = []

for _, row in station_df.iterrows():
    pm25_ranking.append({
        'rank':              int(row['rank']),
        'station_code':      str(row['station_code']),
        'station_name':      str(row['nama_sensor']),
        'kota':              str(row['kota_kabupaten']),
        'provinsi':          str(row['provinsi']),
        'region_group':      str(row['region_group']),
        'karakteristik':     str(row['karakteristik_wilayah']),
        'site_type':         str(row['site_type']),
        'avg_pm25':          float(row['avg_pm25']),
        'pct_exceed_24h':    float(row['pct_exceed_24h']),
        'dominant_ispu_cat': str(row['dominant_ispu_cat']),
        'ispu_color':        str(row['ispu_color']),
        'pct_valid':         float(row['pct_valid']),
        'lat':               float(row['latitude_dd']),
        'lon':               float(row['longitude_dd']),
    })

print('✅ Peringkat Stasiun Disusun:')
print('\n--- TOP 3 STASIUN PALING BERPOLUSI (Worst Quality) ---')
for s in pm25_ranking[:3]:
    print(f"  #{s['rank']} {s['station_code']} ({s['kota']}) — {s['avg_pm25']} µg/m³ [{s['dominant_ispu_cat']}]")

print('\n--- BOTTOM 3 STASIUN PALING BERSIH (Best Quality) ---')
for s in pm25_ranking[-3:]:
    print(f"  #{s['rank']} {s['station_code']} ({s['kota']}) — {s['avg_pm25']} µg/m³ [{s['dominant_ispu_cat']}]")

✅ Peringkat Stasiun Disusun:

--- TOP 3 STASIUN PALING BERPOLUSI (Worst Quality) ---
  #1 AQMS-BJB-01 (Kota Banjarbaru) — 102.14 µg/m³ [Sedang]
  #2 AQMS-SM2-01 (Kota Samarinda) — 88.98 µg/m³ [Sedang]
  #3 AQMS-JMB-02 (Kabupaten Muaro Jambi) — 68.65 µg/m³ [Tidak Sehat]

--- BOTTOM 3 STASIUN PALING BERSIH (Best Quality) ---
  #25 AQMS-BGR-01 (Kabupaten Bogor) — 13.23 µg/m³ [Baik]
  #26 AQMS-PLU-01 (Kabupaten Poso) — 11.26 µg/m³ [Baik]
  #27 AQMS-IDP-01 (Kabupaten Aceh Besar) — 9.34 µg/m³ [Baik]


In [10]:
# =============================================================================
# LANGKAH 9: Data Titik Peta Geospasial Interaktif (Leaflet Map Markers)
# =============================================================================
# Struktur data geospasial untuk menempatkan 27 titik stasiun pemantau di peta Indonesia.
# Setiap marker membawa payload lengkap untuk popup interaktif.
map_stations = []

for _, row in station_df.iterrows():
    map_stations.append({
        'station_code':      str(row['station_code']),
        'station_name':      str(row['nama_sensor']),
        'kota':              str(row['kota_kabupaten']),
        'provinsi':          str(row['provinsi']),
        'region_group':      str(row['region_group']),
        'karakteristik':     str(row['karakteristik_wilayah']),
        'site_type':         str(row['site_type']),
        'lat':               float(row['latitude_dd']),
        'lon':               float(row['longitude_dd']),
        'avg_pm25':          float(row['avg_pm25']),
        'dominant_ispu_cat': str(row['dominant_ispu_cat']),
        'ispu_color':        str(row['ispu_color']),
        'pct_exceed_24h':    float(row['pct_exceed_24h']),
        'rank':              int(row['rank']),
    })

print(f'✅ Berhasil menyiapkan {len(map_stations)} marker geospasial untuk peta interaktif!')

✅ Berhasil menyiapkan 27 marker geospasial untuk peta interaktif!


In [11]:
# =============================================================================
# LANGKAH 10: Ekspor ke JSON Tunggal & Validasi Integritas Data (Sanity Check)
# =============================================================================
# Gabungkan seluruh komponen ke dalam satu kamus (dictionary) utama
final_output = {
    'meta': {
        'dashboard_title':          'National Executive Cockpit • Tab 1',
        'team_name':                'TIM NASI BAKAR CUMI HITAM PAK KRIS',
        'generated_at':             datetime.now(timezone.utc).isoformat(),
        'date_range_start':         '2024-01-01',
        'date_range_end':           '2026-10-14',
        'total_stations':           len(station_df),
        'total_valid_records':      int(len(valid_aqms)),
        'pp22_threshold_24h_ugm3':   PP22_LIMIT_24H,
        'pp22_threshold_annual_ugm3': 15.0,
        'who_guideline_annual_ugm3': 5.0,
    },
    'kpi': {
        'avg_pm25_national':        avg_pm25_national,
        'pct_days_exceed_24h':      pct_days_exceed_national,
        'station_most_critical':    station_most_critical,
        'hospital_burden_index':    hospital_burden_index,
        'hospital_burden_detail': {
            'avg_daily_resp_admissions': round(float(avg_daily_resp_admissions), 1),
            'total_bed_capacity':        int(bed_capacity),
            'total_hospitals':           len(hospitals),
            'days_observed':             int(n_unique_health_days),
        }
    },
    'ispu_donut':   ispu_donut_data,
    'pm25_ranking': pm25_ranking,
    'map_stations': map_stations,
}

# Simpan berkas JSON ke folder dataset/dashboard/
with open(OUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(final_output, f, ensure_ascii=False, indent=2)

# 2. Sinkronkan ke folder dashboard/ untuk deploy portabel & preview lokal instan
DASHBOARD_JSON = PROJECT_ROOT / 'dashboard/tab1_ikhtisar.json'
DASHBOARD_JS   = PROJECT_ROOT / 'dashboard/tab1_data.js'

with open(DASHBOARD_JSON, 'w', encoding='utf-8') as f:
    json.dump(final_output, f, ensure_ascii=False, indent=2)

with open(DASHBOARD_JS, 'w', encoding='utf-8') as f:
    f.write('window.TAB1_DATA = ' + json.dumps(final_output, ensure_ascii=False, indent=2) + ';\n')


size_kb = OUT_FILE.stat().st_size / 1024
print('===========================================================')
print('🎉 SUKSES! File JSON Tab 1 Berhasil Dibuat!')
print(f'   Lokasi File : {OUT_FILE}')
print(f'   Ukuran File : {size_kb:.1f} KB (sangat ringan < 2 MB, sesuai spesifikasi)')
print(f'   Kunci Utama : {list(final_output.keys())}')
print('===========================================================')


🎉 SUKSES! File JSON Tab 1 Berhasil Dibuat!
   Lokasi File : d:\GLRS\dataset\dashboard\tab1_ikhtisar.json
   Ukuran File : 30.7 KB (sangat ringan < 2 MB, sesuai spesifikasi)
   Kunci Utama : ['meta', 'kpi', 'ispu_donut', 'pm25_ranking', 'map_stations']


In [12]:
# =============================================================================
# LANGKAH 11: Sanity Check (Pengecekan Akhir Isi Berkas JSON)
# =============================================================================
# Membaca kembali berkas JSON yang baru disimpan untuk memverifikasi struktur payload
with open(OUT_FILE, 'r', encoding='utf-8') as f:
    verify = json.load(f)

print('✅ Verifikasi JSON:')
print(f"   Judul Metadata     : {verify['meta']['dashboard_title']}")
print(f"   Rata-rata PM2.5    : {verify['kpi']['avg_pm25_national']} µg/m³")
print(f"   Pelanggaran PP 22  : {verify['kpi']['pct_days_exceed_24h']}% hari")
print(f"   Stasiun Terkritis  : {verify['kpi']['station_most_critical']['station_code']} ({verify['kpi']['station_most_critical']['kota']})")
print(f"   Beban Faskes       : {verify['kpi']['hospital_burden_index']}%")
print(f"   Jumlah Item Donut  : {len(verify['ispu_donut'])} kategori")
print(f"   Jumlah Stasiun Rank: {len(verify['pm25_ranking'])} stasiun")
print(f"   Jumlah Titik Peta  : {len(verify['map_stations'])} marker")
print('\nSemua data siap dikonsumsi langsung oleh HTML, ECharts, dan Leaflet pada Tab 1!')

✅ Verifikasi JSON:
   Judul Metadata     : National Executive Cockpit • Tab 1
   Rata-rata PM2.5    : 43.25 µg/m³
   Pelanggaran PP 22  : 17.7% hari
   Stasiun Terkritis  : AQMS-BJB-01 (Kota Banjarbaru)
   Beban Faskes       : 6.43%
   Jumlah Item Donut  : 3 kategori
   Jumlah Stasiun Rank: 27 stasiun
   Jumlah Titik Peta  : 27 marker

Semua data siap dikonsumsi langsung oleh HTML, ECharts, dan Leaflet pada Tab 1!
